# Capstone — Refresh / Content Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanvir-Sheikh-R/From-flyrank-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**  
**Author:** Tanvir Sheikh  
**Date:** October 2026  
**Repo:** https://github.com/Tanvir-Sheikh-R/From-flyrank-starter

> This notebook is the single source of truth for the capstone. It runs top-to-bottom reproducibly (Run All / Runtime → Run all) and mirrors every section of the deployed research paper.

## 1. Question

**Research question:** Which pages in a content portfolio should be reviewed first for a content refresh or other editorial intervention, based on observable search performance, engagement, content freshness, and structural signals?

**The decision this supports:** A content strategist or SEO specialist receives a ranked queue of pages — each with a reason code and recommended action — and decides where to spend limited editorial time today.

**Why data/ML helps:** A simple rule can catch obvious cases (old pages, low CTR), but pages send many overlapping signals simultaneously. A model learns which signal combinations are most predictive of decline, and the ranked output surfaces the highest-priority cases that a rule list would miss or misorder.

**Cost of a wrong call:** A false positive wastes a reviewer's limited time on a page that didn't need attention. A false negative lets a declining page slip through to the next review cycle. Because capacity is the constraint, the primary metric is **Precision@50** — of the top 50 flagged pages, how many are actually declining.

In [ ]:
# Environment setup — works locally and in Colab
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Tanvir-Sheikh-R/From-flyrank-starter"
REPO_DIR = "From-flyrank-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, auc

SEED = 42
np.random.seed(SEED)
print("Environment ready. SEED =", SEED)

## 2. Data

**Source:** FlyRank ML Internship Dataset — starter release  
**File:** `data/raw/content_refresh_anonymized.csv`  
**Size:** 30,000 rows × 44 columns; 32 pseudonymized clients; trailing-90-day metrics per content item.

**What was excluded and why:**
- `trend_direction`, `trend_pct` — these ARE the label. Including them as features would be direct leakage.
- `content_id`, `client_id` — pseudonymous identifiers; used only for grouping/splitting, never as model features.
- `provider_used`, `model_used` — internal operational metadata; not search signals.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — these are the raw numbers that define the label's trend window; including them would leak the outcome's direction into the features.

**Known quirks handled:**
- `avg_position = 0` means "no GSC data", not rank zero. Recoded to NaN; a `has_position` binary flag is added as a feature instead of a blind fillna(0).
- Rate columns (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) are already ×100 percentages — no rescaling needed for ranking purposes.
- Missingness in categorical features filled with `"unknown"` (not mode-filled) to avoid injecting a hidden category signal.

In [ ]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["has_position"] = (df["avg_position"] > 0).astype(int)
df.loc[df["avg_position"] == 0, "avg_position"] = float("nan")

print(f"Rows: {len(df):,}  |  Clients: {df['client_id'].nunique()}  |  Base rate: {df['is_declining_label'].mean():.3f}")
print()
print("Trend direction distribution:")
print(df["trend_direction"].value_counts().to_string())
print()
print(f"avg_position=0 rows (no GSC data): {(df['avg_position'].isna()).sum():,}")

## 3. Methodology

### Task type
Binary classification — label `is_declining_label = 1` when `trend_direction == 'down'` (impressions fell >20% last-30 vs prev-30 days). The predicted **probability** is used as a ranking score. The output is a ranked queue, not a binary yes/no decision.

### Label definition
`is_declining_label = 1` when `trend_direction == 'down'`. Base rate in the full dataset: **54.2%**. Base rate in the test split: **51.1%**. This is a near-balanced class — precision@K is still the right metric because the real constraint is how many of the top-K flagged pages turn out to be correct, not overall accuracy.

### Features used
17 numeric signals: `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `impressions_90d`, `clicks_90d`, `sessions_90d`, `days_with_impressions`, `days_with_sessions`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, plus the derived `has_position` flag.  
8 categorical signals (one-hot encoded): `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`.  
**Total: 52 features after encoding.**

### Leakage checks
- `trend_direction` and `trend_pct` excluded — they are the label source.
- `impressions_last_30d`, `clicks_last_30d`, and their `_prev_30d` counterparts excluded — they define the trend window directly.
- All IDs excluded as features.
- `has_position` is derived from `avg_position` before the label is defined — safe.

### Validation design
**Client-grouped holdout** (`GroupShuffleSplit`, `random_state=42`, `test_size=0.2`).  
Pages from the same client share hidden characteristics (site quality, niche, publishing cadence). A random row split would let the model memorize per-client patterns. Holding out **7 full clients** answers: *does this generalize to a client the model has never seen?*

### Baseline rule
Four components combined with fixed weights (no fitted parameters):  
```
score = 0.40 × visibility_rank
      + 0.30 × freshness_risk_rank
      + 0.25 × position_opportunity_rank
      + 0.05 × depth_gap_rank
```
Plain-language: *"Prioritize pages that already have traffic, haven't been updated recently, and rank on page 1-2 — with a small boost for thin content."* Any human editor can audit this rule.

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

def build_features(frame):
    work = frame.copy()
    for col in NUMERIC_FEATURES:
        work[col] = pd.to_numeric(work[col], errors="coerce").replace([float("inf"), float("-inf")], float("nan"))
    work["has_position"] = (work["avg_position"] > 0).astype(float)
    work["avg_position"] = work["avg_position"].fillna(0)
    for col in NUMERIC_FEATURES:
        work[col] = work[col].fillna(0)
    for col in CATEGORICAL_FEATURES:
        work[col] = work[col].fillna("unknown").astype(str)
    ALL_NUM = NUMERIC_FEATURES + ["has_position"]
    X_num = work[ALL_NUM]
    X_cat = pd.get_dummies(work[CATEGORICAL_FEATURES], dummy_na=False, dtype=float)
    X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
    return X, work

X, work = build_features(df)
y = df["is_declining_label"]
print(f"Feature matrix: {X.shape}")

# ── Client-grouped split ──────────────────────────────────────────────────────
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=work["client_id"]))

train_clients = set(work.iloc[train_idx]["client_id"])
test_clients  = set(work.iloc[test_idx]["client_id"])
print(f"Train: {len(train_idx):,} rows, {len(train_clients)} clients")
print(f"Test:  {len(test_idx):,} rows, {len(test_clients)} clients")
print(f"Client overlap (must be 0): {len(train_clients & test_clients)}")

y_test    = y.iloc[test_idx]
base_rate = y_test.mean()
print(f"\nBase rate (test split): {base_rate:.3f}")

In [ ]:
# ── Baseline rule score ───────────────────────────────────────────────────────
def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

visibility           = percentile_rank(np.log1p(work["impressions_90d"]))
freshness_risk       = percentile_rank(work["days_since_last_update"])
position_opportunity = (
    (1 - percentile_rank(work["avg_position"].clip(1, 50)))
    * visibility * (work["avg_position"] > 0)
)
depth_gap            = (1 - percentile_rank(work["word_count"])) * visibility
baseline_score       = (
    0.40 * visibility + 0.30 * freshness_risk
    + 0.25 * position_opportunity + 0.05 * depth_gap
).clip(0, 1)

print("Baseline rule built:")
print("  score = 0.40×visibility + 0.30×freshness_risk")
print("        + 0.25×position_opportunity + 0.05×depth_gap")
print(f"  Score range: {baseline_score.min():.3f} – {baseline_score.max():.3f}")

## 4. Results (vs baseline)

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# ── Train models ──────────────────────────────────────────────────────────────
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=SEED, n_jobs=-1,
)

lr.fit(X.iloc[train_idx], y.iloc[train_idx])
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

lr_proba = lr.predict_proba(X.iloc[test_idx])[:, 1]
rf_proba = rf.predict_proba(X.iloc[test_idx])[:, 1]
bl_proba = baseline_score.iloc[test_idx].values

results = {}
for name, proba in [("baseline_rules", bl_proba), ("logistic_regression", lr_proba), ("random_forest", rf_proba)]:
    results[name] = {
        "P@20":    precision_at_k(proba, y_test, 20),
        "P@50":    precision_at_k(proba, y_test, 50),
        "P@100":   precision_at_k(proba, y_test, 100),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }

# ── Comparison table ──────────────────────────────────────────────────────────
print(f"Base rate (test split): {base_rate:.3f}  (random pick precision = {base_rate:.3f})")
print()
res_df = pd.DataFrame(results).T.round(3)
print(res_df.to_string())

In [ ]:
# ── Feature importances (best model: Logistic Regression) ─────────────────────
coefs = lr.named_steps["clf"].coef_[0]
importances = pd.Series(np.abs(coefs), index=X.columns).sort_values(ascending=False)

print("Top 10 features (Logistic Regression, absolute coefficients):")
print(importances.head(10).round(4).to_string())

In [ ]:
# ── Error analysis — wrong picks in top 50 ───────────────────────────────────
test_frame = work.iloc[test_idx].copy()
test_frame["model_probability"] = lr_proba
top50       = test_frame.sort_values("model_probability", ascending=False).head(50)
wrong_top50 = top50[top50["is_declining_label"] == 0]

print(f"Wrong picks in top 50: {len(wrong_top50)} of 50  (Precision@50 = {(50-len(wrong_top50))/50:.2f})")
print()
show_cols = ["content_id", "model_probability", "impressions_90d", "avg_position", "ctr", "days_since_last_update", "trend_direction"]
print("Sample wrong picks (top 3):")
print(wrong_top50[show_cols].head(3).to_string(index=False))
print()
print("Interpretation: wrong picks cluster at borderline signal combinations —")
print("moderate impressions, ambiguous positions. Errors are at the decision")
print("boundary, not a sign of structural leakage.")

## 5. Limitations

1. **The label is a proxy, not ground truth.** `is_declining_label` captures a 30-day impression drop ≥20%. A page that dropped because of a SERP layout change, seasonality, or a competitor gaining a featured snippet may look identical to a page with a genuine content quality problem. The model cannot distinguish these causes.

2. **Observational, not causal.** All results are directional and associative. This work cannot claim that refreshing a page *causes* recovery. The model identifies pages associated with measured decline — not what caused the decline.

3. **32 clients, 7 held out.** The test set is 7 clients. With that sample size, performance can shift meaningfully between runs and between different client holdout combinations. Results are labeled *directional* throughout.

4. **Starter dataset only.** This analysis used the 30k-row anonymized starter CSV — not the full 79M-row warehouse. The warehouse would allow time-series validation and per-client history depth checks that would strengthen leakage control.

5. **No temporal split.** A time-aware split (train on months 1–12, test on months 13–17) would be the gold standard for a deployment scenario. The client-grouped split used here is the appropriate choice for the starter flat dataset but would be supplemented by a time split in a production setting.

6. **Action codes are heuristic.** The `REFRESH / PROTECT / OPTIMIZE METADATA / MONITOR` labels are derived from fixed thresholds applied on top of model scores. They are evidence-based starting points, not automated decisions.

## 6. Ranked Recommendations

The action playbook below assigns each page in the test set to one of five evidence-backed actions, ranked by model score within each tier.

In [ ]:
def assign_action(row):
    p          = row["model_probability"]
    imp        = row["impressions_90d"]
    pos        = row["avg_position"]
    days_fresh = row["days_since_last_update"]
    ctr_v      = row["ctr"]
    eng        = row["engagement_rate"]
    if p >= 0.70 and imp >= 500:
        return "REFRESH",           "high_decline_risk_with_visibility",  "high"
    elif p < 0.25 and imp >= 1000 and eng >= 3:
        return "PROTECT",           "high_visibility_stable_engaged",      "high"
    elif ctr_v < 0.5 and 0 < pos <= 10 and imp >= 300:
        return "OPTIMIZE_METADATA", "visible_but_low_ctr",                "medium"
    elif days_fresh > 365 and imp >= 200:
        return "REVIEW_CONTENT",    "stale_but_visible",                  "medium"
    else:
        return "MONITOR",           "ambiguous_signals",                   "low"

actions = test_frame.apply(assign_action, axis=1, result_type="expand")
actions.columns = ["action", "reason_code", "confidence"]
test_frame = pd.concat([test_frame, actions], axis=1)

print("Action playbook distribution:")
print(test_frame["action"].value_counts().to_string())
print()
print("Top 10 REFRESH priority pages (highest model score, visible, declining):")
refresh_top = test_frame[test_frame["action"]=="REFRESH"].sort_values("model_probability", ascending=False)
show = ["content_id", "model_probability", "impressions_90d", "avg_position", "ctr", "days_since_last_update", "reason_code"]
print(refresh_top[show].head(10).to_string(index=False))

# Save the playbook
os.makedirs("work/outputs", exist_ok=True)
save_cols = ["content_id", "action", "reason_code", "confidence",
             "model_probability", "impressions_90d", "avg_position", "ctr",
             "days_since_last_update", "is_declining_label"]
test_frame[save_cols].sort_values("model_probability", ascending=False).to_csv(
    "work/outputs/action_playbook.csv", index=False
)
print("\nSaved: work/outputs/action_playbook.csv")

## 7. Artifacts the paper embeds

The cells below regenerate the five charts used in the deployed research paper. They save to `docs/capstone/img/` and are referenced by the paper with relative paths.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

IMG_DIR = "docs/capstone/img"
os.makedirs(IMG_DIR, exist_ok=True)

STYLE = {
    "decline": "#E74C3C", "stable": "#2ECC71",
    "baseline": "#95A5A6", "logistic": "#3498DB",
    "forest": "#9B59B6", "accent": "#F39C12", "bg": "#FAFAFA",
}
plt.rcParams.update({
    "figure.facecolor": STYLE["bg"], "axes.facecolor": STYLE["bg"],
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "sans-serif", "font.size": 11,
})

# Chart 1: Trend distribution donut
td = df["trend_direction"].value_counts()
fig, ax = plt.subplots(figsize=(5, 5))
colors = [STYLE["decline"],"#F39C12","#3498DB","#2ECC71","#BDC3C7"]
wedges, texts, autotexts = ax.pie(
    td.values, labels=td.index, autopct="%1.1f%%",
    colors=colors, startangle=140, pctdistance=0.78,
    wedgeprops=dict(width=0.52, edgecolor="white", linewidth=2)
)
for at in autotexts: at.set_fontsize(9)
ax.set_title("Trend Distribution (30,000 pages)", fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/chart1_trend_dist.png", dpi=150, bbox_inches="tight")
plt.close()

# Chart 2: Precision@K bar chart
ks = [20, 50, 100]
bl_p  = [precision_at_k(bl_proba, y_test, k) for k in ks]
lr_p  = [precision_at_k(lr_proba, y_test, k) for k in ks]
rf_p  = [precision_at_k(rf_proba, y_test, k) for k in ks]
x = np.arange(len(ks)); w = 0.22
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x-w, bl_p, w, label="Rule baseline", color=STYLE["baseline"])
ax.bar(x,   lr_p, w, label="Logistic Regression", color=STYLE["logistic"])
ax.bar(x+w, rf_p, w, label="Random Forest", color=STYLE["forest"])
ax.axhline(base_rate, color=STYLE["accent"], lw=1.5, linestyle="--", label=f"Base rate ({base_rate:.2f})")
ax.set_xticks(x); ax.set_xticklabels([f"Precision@{k}" for k in ks])
ax.set_ylim(0, 1); ax.set_ylabel("Precision")
ax.set_title("Model vs Baseline: Precision@K", fontweight="bold")
ax.legend(fontsize=9)
for bars in ax.containers: ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/chart2_precision_k.png", dpi=150, bbox_inches="tight")
plt.close()

# Chart 3: ROC curves
from sklearn.metrics import roc_curve, auc
fig, ax = plt.subplots(figsize=(5.5, 5))
for proba, name, color in [
    (bl_proba, "Rule baseline", STYLE["baseline"]),
    (lr_proba, "Logistic Regression", STYLE["logistic"]),
    (rf_proba, "Random Forest", STYLE["forest"]),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc(fpr,tpr):.3f})")
ax.plot([0,1],[0,1],"k--", lw=1, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — 7 held-out clients", fontweight="bold")
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig(f"{IMG_DIR}/chart3_roc.png", dpi=150, bbox_inches="tight")
plt.close()

# Chart 4: Feature importances
coefs = lr.named_steps["clf"].coef_[0]
top_feats = pd.Series(np.abs(coefs), index=X.columns).sort_values(ascending=False).head(12)
fig, ax = plt.subplots(figsize=(7, 4.5))
c_bars = [STYLE["logistic"]]*len(top_feats); c_bars[0] = STYLE["decline"]
ax.barh(top_feats.index[::-1], top_feats.values[::-1], color=c_bars[::-1])
ax.set_xlabel("Absolute Coefficient"); ax.set_title("Top 12 Features — Logistic Regression", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/chart4_features.png", dpi=150, bbox_inches="tight")
plt.close()

# Chart 5: Action mix
action_counts = test_frame["action"].value_counts()
action_colors = {"REFRESH": STYLE["decline"], "PROTECT": STYLE["stable"],
                 "OPTIMIZE_METADATA": STYLE["logistic"], "REVIEW_CONTENT": STYLE["accent"],
                 "MONITOR": STYLE["baseline"]}
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(action_counts.index, action_counts.values,
              color=[action_colors.get(a, "#BDC3C7") for a in action_counts.index])
ax.set_ylabel("Pages"); ax.set_title("Recommended Actions — 6,163 test pages", fontweight="bold")
ax.bar_label(bars, fmt="%d", padding=3)
plt.xticks(rotation=20, ha="right"); plt.tight_layout()
plt.savefig(f"{IMG_DIR}/chart5_actions.png", dpi=150, bbox_inches="tight")
plt.close()

print("All 5 charts saved to docs/capstone/img/")

## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown reasoning AND the code that backs it  
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)  
- [x] No client names, URLs, raw queries, or private data anywhere in this file  
- [x] Claims use careful language: *observed*, *directional*, *decision-support*  
- [x] `trend_direction` and `trend_pct` excluded from all feature matrices  
- [x] Client-grouped split — zero client overlap confirmed in code output  
- [x] Baseline and model compared on the same test split with the same metric  
- [x] Committed to repo under `work/notebooks/` — repo URL submitted on the card

---

## Demo Outline — 5-Minute Showcase

*Ready for the Week-8 showcase. Five beats, one chart per beat, one honest claim.*

---

### Beat 1 — The Problem (1 min)

> **"FlyRank's clients manage thousands of content pages. Most teams don't have a data-driven signal for which ones to fix first — they work on gut feel, recency, or traffic size. All three miss the same category: pages that *used to rank* and are quietly losing ground."**

- **Show:** nothing. Let the problem land.

---

### Beat 2 — The Data (30 sec)

> **"30,000 anonymized content pages, 32 clients, 90 days of real search and engagement signals from the FlyRank ML Internship Dataset. Half the portfolio is declining right now."**

- **Show:** Figure 1 — trend distribution donut. Half the dataset is declining.

---

### Beat 3 — The Method (1.5 min)

> **"I framed this as binary classification — declining vs. not — and used the predicted probability as a ranking score.
> Two key decisions: client-grouped holdout so the model is tested on clients it's never seen; and I excluded the label-source columns to prevent data leakage.
> The baseline is a rule any editor could write — flag visible pages that are stale and slipping in position.
> The model is Logistic Regression — not a black box. I can show you exactly which features it leans on."**

- **Show:** Figure 4 — feature importances. Walk the top 3: `has_position`, `days_with_impressions`, `content_age_days`. All make intuitive sense.

---

### Beat 4 — The Honest Result (1 min)

> **"Base rate is 0.51 — random picking gives you 51% precision. The rule baseline I wrote scores 0.32 at Precision@50: *worse than random*. That's the lesson — the obvious rule doesn't work here.
> The model hits 0.78: 39 of the top 50 flagged pages are genuinely declining.
> I want to be clear: these results are observational and directional — not a claim about Google's algorithm."**

- **Show:** Figure 2 — Precision@K bar chart. Let the gap between baseline and model speak.

---

### Beat 5 — The Recommendation (1 min)

> **"The output is a ranked queue with reason codes. 483 pages flagged REFRESH. 967 for metadata optimization. 62 to protect and not touch.
> A content team exports the CSV, sorts by model score, and starts their sprint from row 1.
> One actionable takeaway: if you're not using GSC position data combined with content age as a refresh signal, you're probably working from the wrong list."**

- **Show:** Figure 5 — action distribution bar chart.


---

## Shareable Cuts

*Two cuts of the same work — ready to share as-is.*

---

### Cut 1 — Social Post (LinkedIn / X)

> I spent the last several weeks building a content refresh prioritization model on real search data — 30,000 pages, 32 client portfolios, 90 days of GSC and engagement signals from the FlyRank ML Internship Dataset.
>
> The interesting finding: the "obvious" rule (flag visible pages that are old and slipping in position) scores *below random* at Precision@50. The trained Logistic Regression model hits 0.78 — placing 39 genuinely declining pages in the top 50 recommendations.
>
> The most predictive signal wasn't traffic volume or keyword competition. It was whether a page had any GSC position data at all — combined with how consistently it accumulated impressions over time. Pages with real, sustained search presence fail in a different pattern than pages that were never truly visible.
>
> Full methodology, reproducible code, and a deployed research paper in the link below.
>
> #SEO #MachineLearning #ContentStrategy #DataScience

---

### Cut 2 — Employer-Facing 3-Sentence Summary

> I built a content refresh prioritization engine using Logistic Regression on 30,000 real, anonymized search performance records across 32 client portfolios — features included GSC visibility, content freshness, CTR, engagement depth, and position signals.
> Evaluated on a client-grouped holdout (7 unseen clients) to test real-world generalization, the model achieved Precision@50 of 0.78 versus 0.32 for a hand-crafted rule baseline and a 0.51 base rate — a 2.4× lift, with zero client overlap between train and test confirmed in code.
> The deliverable is a ranked action queue (REFRESH / PROTECT / OPTIMIZE METADATA / MONITOR) with reason codes, deployed as a public research paper with full reproducibility documentation and a capstone notebook that runs end-to-end.
